In [ ]:
import subprocess, sys, os

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xy"], check=True)

WIDGETS_OK = True
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    WIDGETS_OK = False

import numpy as np
import pandas as pd
import xy
from IPython.display import display, HTML

print("xy", xy.__version__, "| live widgets:", WIDGETS_OK)

def render(chart, note=""):
    if note:
        display(HTML(f"<h3 style='font:600 15px system-ui;margin:18px 0 6px'>{note}</h3>"))
    try:
        display(chart)
    except Exception:
        display(HTML(chart.to_html()))
    return chart

rng = np.random.default_rng(7)

days    = np.arange(180)
trend   = 200 + 0.9 * days + 18 * np.sin(days / 9.0)
revenue = trend + rng.normal(0, 12, days.size)
sigma   = 10 + 6 * np.abs(np.sin(days / 15.0))
conv    = 0.06 + 0.02 * np.sin(days / 21.0) + rng.normal(0, 0.003, days.size)
peak    = int(np.argmax(revenue))

layered = xy.chart(
    xy.error_band(days, revenue - 1.96 * sigma, revenue + 1.96 * sigma,
                  name="95% band", color="#7c3aed", opacity=0.16),
    xy.line(days, revenue, name="Revenue", color="#7c3aed", width=2.5,
            curve="smooth"),
    xy.scatter(days[::12], revenue[::12], name="Weekly check", color="#7c3aed",
               size=7, stroke="#ffffff", stroke_width=1.5),
    xy.line(days, conv, name="Conversion", color="#f59e0b", width=2,
            dash="dashed", y_axis="y2"),

    xy.x_axis(label="Day", grid=True),
    xy.y_axis(label="Revenue (k)", grid=True, format=",.0f"),
    xy.y_axis(id="y2", label="Conversion", side="right", grid=False, format=".1%"),

    xy.x_band(120, 150, text="Campaign", color="#22c55e", opacity=0.10),
    xy.hline(float(revenue.mean()), text="mean", color="#94a3b8"),
    xy.callout(float(days[peak]), float(revenue[peak]), "peak", dx=-60, dy=-40),

    xy.legend(loc="upper left", ncols=2, toggle=True),
    xy.tooltip(title="Day", format={"y": ",.1f"}),
    xy.modebar(True),
    xy.theme(palette=["#7c3aed", "#f59e0b"], grid_color="#e6e6ef"),

    title="Layered composition · dual axes · annotations",
    width=900, height=440, crosshair=True,
)
render(layered, "1 · Composition model")

In [ ]:
n = 4000
df = pd.DataFrame({
    "x":      rng.normal(0, 1, n),
    "noise":  rng.normal(0, 1, n),
    "region": rng.choice(["North", "South", "East", "West"], n),
})
df["y"]   = 2.1 * df["x"] + df["noise"] * 0.9
df["mag"] = np.abs(df["y"])

render(xy.scatter_chart(
    xy.scatter("x", "y", color="mag", colormap="plasma",
               size=5, opacity=0.7, color_domain=(0, 6)),
    xy.colorbar(title="|y|"),
    xy.x_axis(label="x"), xy.y_axis(label="y"),
    data=df, title="Columns resolved by name", width=760, height=420,
), "2 · DataFrame-driven channels")

render(xy.facet_chart(
    xy.scatter("x", "y", color="#0ea5e9", size=4, opacity=0.6),
    by="region", data=df, cols=2,
    share_x=True, share_y=True, link=True, link_select=True,
    width=760, height=220, gap=12, title="Faceted by region",
), "3 · Facets with linked axes")

N = 1_500_000
r     = 6.0 * rng.beta(1.2, 3.0, N)
theta = 2.9 * np.log1p(r) + rng.integers(0, 4, N) * (np.pi / 2) + rng.normal(0, 0.05, N)

big = xy.scatter_chart(
    xy.scatter(r * np.cos(theta), r * np.sin(theta),
               color=np.exp(-r / 2.2), colormap="magma_r",
               density=True,
               size=2.5, opacity=0.85,
               zoom_size_factor=2.6, zoom_opacity=0.95),
    xy.colorbar(title="density"),
    title=f"{N:,} points · drag to pan, scroll to zoom",
    width=760, height=520, zoom=True, pan=True, wheel_zoom=True,
)
render(big, "4 · Million-point density surface")

mem = big.memory_report()
print(f"canonical f64 held in Python : {mem['canonical_bytes']/1e6:.1f} MB")
print(f"bytes sent for first paint   : {mem['transport_bytes_first_paint']/1e6:.2f} MB "
      f"({mem['transport_bytes_per_point']:.3f} B/point)")
print(f"compute backend              : {mem['backend']}")

In [ ]:
sel = big.select_range(-1.0, 1.0, -1.0, 1.0)
sx, sy = sel.xy(0)
print(f"\nselect_range hit {len(sel):,} rows; x array {sx.shape}")
print("first rows:", sel.rows(limit=2))
print("pick(trace=1, index=10):", layered.pick(1, 10))

def on_select(selection):
    xs, ys = selection.xy(0)
    print(f"[callback] {len(selection):,} rows selected, mean y = {ys.mean():.3f}")

def on_view_change(payload):
    print("[callback] viewport:", payload)

render(xy.scatter_chart(
    xy.scatter("x", "y", color="#ef4444", size=5, opacity=0.7),
    data=df, select=True, on_select=on_select, on_view_change=on_view_change,
    title="Shift-drag a box → payload lands in Python",
    width=760, height=380,
), "5 · Selections routed back to the kernel")

stream = xy.line_chart(
    xy.line([0.0], [0.0], color="#10b981", width=2, name="live"),
    xy.x_axis(label="t"), xy.y_axis(label="value", domain=(-3, 3)),
    title="Streaming via chart.append()", width=760, height=320,
)
render(stream, "6 · Streaming")

import time
for k in range(1, 60):
    t = k / 3.0
    stream.append(0, [t], [float(np.sin(t) + rng.normal(0, 0.08))])
    time.sleep(0.03)

In [ ]:
print("\navailable slots:", ", ".join(sorted(xy.CHART_DOM_SLOTS)))

CSS = """
.xy-card {background:#fafaf9;border:1px solid #e7e5e4;border-radius:16px;padding:10px}
.xy-title{font:600 16px/1.2 ui-sans-serif;letter-spacing:-.01em;color:#1c1917}
.xy-tip  {border-radius:10px;background:#1c1917;color:#fafaf9}
"""
display(HTML(f"<style>{CSS}</style>"))

styled = xy.line_chart(
    xy.line(days, revenue, color="#111827", width=2,
            animation=xy.animation(duration=700,
                                   easing=xy.spring(stiffness=180, damping=22))),
    xy.x_axis(label="Day"), xy.y_axis(label="Revenue"),
    title="Slot-addressed styling",
    class_name="xy-card",
    class_names={"title": "xy-title", "tooltip": "xy-tip"},
    styles={"canvas": {"border-radius": "12px"}},
    width=760, height=360,
)
render(styled, "7 · CSS slots, tokens, spring animation")

def _fit(cols):
    x = np.asarray(cols["x"], float); y = np.asarray(cols["y"], float)
    b, a = np.polyfit(x, y, 1)
    order = np.argsort(x); xs = x[order]
    fit   = a + b * xs
    resid = float(np.std(y - (a + b * x)))
    return {"x": xs, "y": y[order], "fit": fit,
            "lo": fit - 1.96 * resid, "hi": fit + 1.96 * resid}

def _build(ctx):
    color = ctx.options.get("color", "#2563eb")
    c, nm = ctx.columns, (ctx.name or "trend")
    return [
        xy.error_band(c["x"], c["lo"], c["hi"], color=color, opacity=0.18, name=f"{nm} CI"),
        xy.line(c["x"], c["fit"], color=color, width=2.5, name=nm),
    ]

if "trendline" not in xy.registered_marks():
    xy.register_mark(xy.MarkPlugin(name="trendline", build=_build,
                                   columns=("x", "y"), calc=_fit,
                                   doc="OLS fit with a 95% band."))

render(xy.chart(
    xy.scatter("x", "y", color="#94a3b8", size=4, opacity=0.5, name="observations"),
    xy.mark("trendline", x="x", y="y", color="#e11d48", name="OLS"),
    xy.legend(loc="upper left"),
    data=df, title="Third-party mark kind", width=760, height=400,
), "8 · Custom mark plugin")

In [1]:
import xy.pyplot as plt
t = np.linspace(0, 10, 400)
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(t, np.sin(t), "r--", label="sin")
ax.plot(t, np.cos(t), label="cos")
ax.set_xlabel("t"); ax.set_ylabel("amplitude"); ax.set_title("xy.pyplot compatibility")
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

os.makedirs("out", exist_ok=True)
layered.to_html("out/chart.html")
layered.to_svg("out/chart.svg")
layered.to_png("out/chart.png", scale=2)
for f in ("chart.html", "chart.svg", "chart.png"):
    print(f"out/{f}: {os.path.getsize('out/'+f)/1024:.0f} KB")

print("\n✅ tutorial complete")

xy 0.0.5 | live widgets: True


<iframe class="xy-notebook-frame" sandbox="allow-scripts" width="760" height="476" style="display:block;width:100%;max-width:760px;height:476px;margin-left:8px;border:0;background:transparent" srcdoc="<!doctype html>
<html><head><meta charset="utf-8">
<meta http-equiv="Content-Security-Policy" content="default-src 'none'; script-src 'unsafe-inline'; style-src 'unsafe-inline'; img-src data:; font-src data:; connect-src 'none'; worker-src blob:; object-src 'none'; base-uri 'none'; form-action 'none'">
<title>Faceted by region</title>
<style>
.xy-facet-document{margin:0;width:100%;min-height:100%;font-family:system-ui,sans-serif;background:#fff;}
.xy-facet-document .xy-facet-title{height:24px;line-height:24px;font:600 14px system-ui,sans-serif;margin:0;text-align:center;color:#1e293b;}
.xy-facet-document .xy-facet-grid{display:grid;grid-template-columns:repeat(2, minmax(0, 1fr));gap:12px;}
.xy-facet-document .xy-facet-panel{min-width:0;}
</style>
</head><body class="xy-facet-document">
<div class="xy-facet-title">Faceted by region</div><div class="xy-facet-grid" id="xy-facet-grid"></div>
<script>var xy=(function(e){Object.defineProperties(e,{__esModule:{value:!0},[Symbol.toStringTag]:{value:`Module`}});var t=`xBuf.yBuf.cBuf.sBuf.selBuf.baseBuf.x0Buf.x1Buf.x2Buf.y0Buf.y1Buf.y2Buf.t0Buf.t1Buf.posBuf.value1Buf.value0Buf.rgbaBuf.rgba2Buf.styleBuf.strokeBuf.radiusBuf.dBuf._lenBuf._segmentDashOffsetBuf._segmentDashDirBuf._transitionPrevXBuf._transitionPrevYBuf._transitionPrevPosBuf._transitionPrevValue1Buf._transitionPrevValue0Buf`.split(`.`),n=[88,89,66,70],r=1,i=24,a=8,o=Object.freeze({maxFrameBytes:512*1024*1024,maxMetadataBytes:8*1024*1024,maxBuffers:4096,maxBufferBytes:256*1024*1024});function s(e,t=`buffer`){if(e instanceof ArrayBuffer)return new Uint8Array(e);if(ArrayBuffer.isView(e))return new Uint8Array(e.buffer,e.byteOffset,e.byteLength);throw TypeError(`${t} must be an ArrayBuffer or ArrayBuffer view`)}function c(e){let t=s(e,`chart payload`);return t.byteOffset%4==0?t:new Uint8Array(t)}function l(e,t){let n=e?.columns;if(!Array.isArray(n))return!1;let r=e=>e.dtype===`u8`?1:4;return e.buffer_layout===`split`?Array.isArray(t)?n.every(e=>{if(!Number.isInteger(e.buf))return!0;let n=t[e.buf];return!!n&&e.len*r(e)<=n.byteLength}):!1:Array.isArray(t)||!t?!1:n.every(e=>e.byte_offset+e.len*r(e)<=t.byteLength)}function u(e,t){if(e.buffer_layout===`split`){if(!Array.isArray(t))throw Error(`xy: spec says buffer_layout=split but the transport delivered one buffer`);return t.map(c)}if(Array.isArray(t))throw Error(`xy: transport delivered a buffer list but the spec is not split-layout`);return c(t)}function d(e,t){let n=o[t],r=e&&e[t]!=null?e[t]:n;if(!Number.isSafeInteger(r)||r<=0)throw RangeError(`${t} must be a positive safe integer`);return r}function f(e){return Math.ceil(e/a)*a}function p(e,t,n){let r=e.getBigUint64(t,!0);if(r>BigInt(2**53-1))throw RangeError(`${n} exceeds JavaScript's safe integer range`);return Number(r)}function m(e,t,n,r){if(n>e.byteLength)throw RangeError(`truncated ${r} padding`);for(let i=t;i<n;i++)if(e[i]!==0)throw RangeError(`non-zero ${r} padding`)}function h(e,t=null){let o=s(e,`frame body`),c=d(t,`maxFrameBytes`),l=d(t,`maxMetadataBytes`),u=d(t,`maxBuffers`),h=d(t,`maxBufferBytes`);if(l>c)throw RangeError(`maxMetadataBytes cannot exceed maxFrameBytes`);if(h>c)throw RangeError(`maxBufferBytes cannot exceed maxFrameBytes`);if(o.byteOffset%a!==0)throw RangeError(`frame body must start on an 8-byte boundary`);if(o.byteLength>c)throw RangeError(`frame length ${o.byteLength} exceeds limit ${c}`);if(o.byteLength<i)throw RangeError(`truncated frame header`);let g=new DataView(o.buffer,o.byteOffset,o.byteLength);for(let e=0;e<n.length;e++)if(g.getUint8(e)!==n[e])throw RangeError(`invalid frame magic`);let _=g.getUint8(4);if(_!==r)throw RangeError(`unsupported frame version ${_}`);let v=g.getUint8(5);if(v!==0)throw RangeError(`unsupported frame flags 0x${v.toString(16)}`);let y=g.getUint16(6,!0);if(y!==i)throw Rang

canonical f64 held in Python : 24.0 MB
bytes sent for first paint   : 1.08 MB (0.723 B/point)
compute backend              : native

select_range hit 665,567 rows; x array (665567,)
first rows: [{'trace': 0, 'index': 4, 'x': 0.8330611227390398, 'y': 0.14570090499531815, 'x_kind': 'float', 'y_kind': 'float', 'color_value': 0.6808508088937596}, {'trace': 0, 'index': 5, 'x': 0.008380729758620429, 'y': -0.05678062679024765, 'x_kind': 'float', 'y_kind': 'float', 'color_value': 0.9742483840008147}]
pick(trace=1, index=10): {'trace': 1, 'index': 10, 'x': 10.0, 'y': 231.0095642207616, 'x_kind': 'float', 'y_kind': 'float'}



available slots: annotation_label, axis_title, badge, badge_item, canvas, chrome, colorbar, colorbar_bar, colorbar_tick, colorbar_title, crosshair_x, crosshair_y, labels, legend, legend_item, legend_label, legend_swatch, legend_title, modebar, modebar_button, root, selection, tick_label, title, tooltip, tooltip_label, tooltip_row, tooltip_title, tooltip_value


out/chart.html: 413 KB
out/chart.svg: 25 KB
out/chart.png: 399 KB

✅ tutorial complete
